# Partitioning & Bucketing in Apache Spark
**Author : Mukesh Date : 06-Dec-2025**
## Overview
This notebook demonstrates:
1. **Partitioning** - Organizing data into directories based on column values
2. **Bucketing** - Distributing data into fixed number of buckets based on hash of column values
3. **Performance comparisons** between partitioned, bucketed, and non-partitioned data

## Dataset
We'll use 5GB of Indian citizen data with fields:
- Aadhar Number, Name, State, Age, Gender, City, Occupation, etc.



## 1. Setup Spark Session

In [1]:
# COPY THIS CODE INTO THE FIRST CELL OF THE JUPYTER NOTEBOOK
# This fixes the SPARK_HOME and PYTHONPATH issues

import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, sum as spark_sum, expr
import time

# Unset conflicting environment variables
os.environ.pop('SPARK_HOME', None)
os.environ.pop('PYTHONPATH', None)

# Java 11 compatibility options
java_opts = "--add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED"
os.environ['PYSPARK_SUBMIT_ARGS'] = f'--driver-java-options "{java_opts}" pyspark-shell'

# Create Spark session
spark = SparkSession.builder \
    .appName("PartitioningBucketingDemo") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.extraJavaOptions", java_opts) \
    .config("spark.executor.extraJavaOptions", java_opts) \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print("Spark session created successfully!")


25/12/08 09:08:30 WARN Utils: Your hostname, mukeshs-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.29.221 instead (on interface en0)
25/12/08 09:08:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/08 09:08:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.3
Spark session created successfully!


## 2. Load the Generated Data

In [2]:
# Load the Indian citizen data
data_path = "indian_citizens_data"

df = spark.read.parquet(data_path)

print(f"Total records: {df.count():,}")
print("\nSchema:")
df.printSchema()

print("\nSample data:")
df.select("aadhar_number", "full_name", "state", "age", "gender", "occupation").show(10, truncate=False)

Total records: 1,073,741

Schema:
root
 |-- aadhar_number: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- state: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- city: string (nullable = true)
 |-- pincode: string (nullable = true)
 |-- mobile_number: string (nullable = true)
 |-- email: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- occupation: string (nullable = true)
 |-- annual_income: integer (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- full_name: string (nullable = true)


Sample data:
+-------------+---------------+--------------+---+------+-------------------+
|aadhar_number|full_name      |state         |age|gender|occupation         |
+-------------+---------------+--------------+---+------+-------------------+
|96075174661  |Isha Verma     |Bihar         |24 |Female|Government Employee|
|38468043297  |Anil

## 3. Data Statistics

In [3]:
print("State Distribution:")
df.groupBy("state").count().orderBy("count", ascending=False).show(20)

print("\nGender Distribution:")
df.groupBy("gender").count().show()

print("\nOccupation Distribution:")
df.groupBy("occupation").count().orderBy("count", ascending=False).show()

State Distribution:
+--------------+------+
|         state| count|
+--------------+------+
|   Maharashtra|282566|
|         Bihar|258655|
| Uttar Pradesh|189929|
|   West Bengal|178432|
|Madhya Pradesh| 98777|
|    Tamil Nadu| 43862|
|     Rajasthan| 15745|
|     Karnataka|  4534|
|       Gujarat|  1039|
|Andhra Pradesh|   180|
|        Odisha|    22|
+--------------+------+


Gender Distribution:
+------+------+
|gender| count|
+------+------+
|Female|526554|
|  Male|547187|
+------+------+


Occupation Distribution:
+-------------------+------+
|         occupation| count|
+-------------------+------+
|             Doctor|240075|
|            Teacher|227515|
|     Business Owner|199926|
|  Software Engineer|161061|
|Government Employee|135199|
|             Farmer| 71342|
|            Student| 28917|
|          Homemaker|  8238|
|              Other|  1468|
+-------------------+------+



## 4. Partitioning

### What is Partitioning?
Partitioning organizes data into separate directories based on column values. This is useful for:
- **Partition pruning**: Skip reading irrelevant partitions
- **Faster queries**: When filtering by partition columns
- **Data organization**: Logical grouping of data

### Example: Partition by State

In [4]:
# Write data partitioned by state
partitioned_path = "indian_citizens_partitioned_by_state"

print("Writing partitioned data...")
start_time = time.time()

df.write \
    .mode("overwrite") \
    .partitionBy("state") \
    .parquet(partitioned_path)

write_time = time.time() - start_time
print(f"Partitioned data written in {write_time:.2f} seconds")
print(f"Location: {partitioned_path}")

Writing partitioned data...


Partitioned data written in 2.83 seconds
Location: indian_citizens_partitioned_by_state


In [5]:
# Check the directory structure
import subprocess
result = subprocess.run(['ls', '-lh', partitioned_path], capture_output=True, text=True)
print("Partition directories:")
print(result.stdout[:1000])  # Show first 1000 chars

Partition directories:
total 0
-rw-r--r--@  1 mukesh  staff     0B Dec  8 09:08 _SUCCESS
drwxr-xr-x@ 18 mukesh  staff   576B Dec  8 09:08 state=Andhra Pradesh
drwxr-xr-x@ 18 mukesh  staff   576B Dec  8 09:08 state=Bihar
drwxr-xr-x@ 18 mukesh  staff   576B Dec  8 09:08 state=Gujarat
drwxr-xr-x@ 18 mukesh  staff   576B Dec  8 09:08 state=Karnataka
drwxr-xr-x@ 18 mukesh  staff   576B Dec  8 09:08 state=Madhya Pradesh
drwxr-xr-x@ 18 mukesh  staff   576B Dec  8 09:08 state=Maharashtra
drwxr-xr-x@ 16 mukesh  staff   512B Dec  8 09:08 state=Odisha
drwxr-xr-x@ 18 mukesh  staff   576B Dec  8 09:08 state=Rajasthan
drwxr-xr-x@ 18 mukesh  staff   576B Dec  8 09:08 state=Tamil Nadu
drwxr-xr-x@ 18 mukesh  staff   576B Dec  8 09:08 state=Uttar Pradesh
drwxr-xr-x@ 18 mukesh  staff   576B Dec  8 09:08 state=West Bengal



### Performance Test: Partitioned vs Non-Partitioned

In [6]:
# Load partitioned data
df_partitioned = spark.read.parquet(partitioned_path)

# Query 1: Filter by state (benefits from partitioning)
print("="*80)
print("QUERY 1: Filter by State = 'Maharashtra'")
print("="*80)

# Non-partitioned
start_time = time.time()
result1 = df.filter(col("state") == "Maharashtra").count()
time_non_partitioned = time.time() - start_time
print(f"Non-partitioned: {result1:,} records in {time_non_partitioned:.2f} seconds")

# Partitioned
start_time = time.time()
result2 = df_partitioned.filter(col("state") == "Maharashtra").count()
time_partitioned = time.time() - start_time
print(f"Partitioned: {result2:,} records in {time_partitioned:.2f} seconds")

speedup = time_non_partitioned / time_partitioned if time_partitioned > 0 else 0
print(f"\n⚡ Speedup: {speedup:.2f}x faster with partitioning!")

QUERY 1: Filter by State = 'Maharashtra'
Non-partitioned: 282,566 records in 0.24 seconds
Partitioned: 282,566 records in 0.07 seconds

⚡ Speedup: 3.50x faster with partitioning!


In [7]:
# Query 2: Aggregation by state
print("="*80)
print("QUERY 2: Average income by State")
print("="*80)

# Non-partitioned
start_time = time.time()
result1 = df.groupBy("state").agg(avg("annual_income").alias("avg_income")).collect()
time_non_partitioned = time.time() - start_time
print(f"Non-partitioned: {time_non_partitioned:.2f} seconds")

# Partitioned
start_time = time.time()
result2 = df_partitioned.groupBy("state").agg(avg("annual_income").alias("avg_income")).collect()
time_partitioned = time.time() - start_time
print(f"Partitioned: {time_partitioned:.2f} seconds")

speedup = time_non_partitioned / time_partitioned if time_partitioned > 0 else 0
print(f"\n⚡ Speedup: {speedup:.2f}x")

QUERY 2: Average income by State
Non-partitioned: 0.35 seconds
Partitioned: 0.19 seconds

⚡ Speedup: 1.83x


## 5. Multi-Level Partitioning

Partition by multiple columns for finer-grained organization.

In [8]:
# Partition by state and gender
multi_partitioned_path = "indian_citizens_partitioned_by_state_gender"

print("Writing multi-level partitioned data...")
start_time = time.time()

df.write \
    .mode("overwrite") \
    .partitionBy("state", "gender") \
    .parquet(multi_partitioned_path)

write_time = time.time() - start_time
print(f"Multi-level partitioned data written in {write_time:.2f} seconds")

Writing multi-level partitioned data...


Multi-level partitioned data written in 2.65 seconds


In [9]:
# Load and query multi-partitioned data
df_multi_partitioned = spark.read.parquet(multi_partitioned_path)

print("Query: Female citizens in Maharashtra")
start_time = time.time()
result = df_multi_partitioned.filter(
    (col("state") == "Maharashtra") & (col("gender") == "Female")
).count()
query_time = time.time() - start_time

print(f"Result: {result:,} records in {query_time:.2f} seconds")
print("\n✅ Multi-level partitioning allows skipping even more data!")

Query: Female citizens in Maharashtra
Result: 138,590 records in 0.08 seconds

✅ Multi-level partitioning allows skipping even more data!


## 6. Bucketing

### What is Bucketing?
Bucketing distributes data into a fixed number of buckets based on hash of column values. Benefits:
- **Efficient joins**: Pre-shuffled data for join operations
- **Consistent distribution**: Fixed number of files
- **Sampling**: Easy to sample data

### Example: Bucket by State

In [10]:
# Create bucketed table
bucketed_table = "indian_citizens_bucketed1"

print("Writing bucketed data...")
start_time = time.time()

df.write \
    .mode("overwrite") \
    .bucketBy(20, "state") \
    .sortBy("state", "age") \
    .mode("overwrite") \
    .saveAsTable(bucketed_table)

write_time = time.time() - start_time
print(f"Bucketed data written in {write_time:.2f} seconds")
print(f"Table: {bucketed_table}")
print(f"Buckets: 20")
print(f"Bucketed by: state")
print(f"Sorted by: state, age")

Writing bucketed data...


Bucketed data written in 2.03 seconds
Table: indian_citizens_bucketed1
Buckets: 20
Bucketed by: state
Sorted by: state, age


In [11]:
# Load bucketed data
df_bucketed = spark.table(bucketed_table)

print("Bucketed data loaded")
df_bucketed.show(10)

Bucketed data loaded
+-------------+----------+----------+-----------+---+------+--------+-------+-------------+--------------------+-------------+-----------------+-------------+-----------------+-----------------+
|aadhar_number|first_name| last_name|      state|age|gender|    city|pincode|mobile_number|               email|date_of_birth|       occupation|annual_income|registration_date|        full_name|
+-------------+----------+----------+-----------+---+------+--------+-------+-------------+--------------------+-------------+-----------------+-------------+-----------------+-----------------+
|  13215454419|      Ritu|      Nair|Maharashtra| 18|  Male|City_173| 191572|   9583382612| ritu.nair@email.com|   2007-07-23|          Teacher|      1342309|       2021-06-12|        Ritu Nair|
|  85461963323|     Karan|     Menon|Maharashtra| 18|Female|City_409| 208131|    753601763|karan.menon@email...|   2007-03-05|           Doctor|      1360871|       2025-01-12|      Karan Menon|
|  8

## 7. Bucketing Performance: Join Operations

Bucketing shines when performing joins on the bucketed column.

In [12]:
# Create a smaller lookup table (state statistics)
state_stats = df.groupBy("state").agg(
    count("*").alias("total_citizens"),
    avg("annual_income").alias("avg_income"),
    avg("age").alias("avg_age")
)

print("State Statistics:")
state_stats.show(10)

State Statistics:
+--------------+--------------+------------------+------------------+
|         state|total_citizens|        avg_income|           avg_age|
+--------------+--------------+------------------+------------------+
|     Karnataka|          4534|1099177.1532862815| 53.12726069695633|
|        Odisha|            22|          969730.5| 54.95454545454545|
|    Tamil Nadu|         43862|1100070.1676394145| 53.47774839268615|
|Andhra Pradesh|           180|1111579.2777777778| 53.90555555555556|
|Madhya Pradesh|         98777|1100775.1617684278| 53.54926754203914|
|       Gujarat|          1039|1094915.8922040423| 53.29355149181906|
|     Rajasthan|         15745|1094844.5034614163| 53.95547792950143|
|   Maharashtra|        282566|1097398.1165851518| 53.55917909444165|
|   West Bengal|        178432|1101681.1655196378|53.546028739239595|
|         Bihar|        258655|1100763.5156753205|53.458487174034914|
+--------------+--------------+------------------+------------------+
on

In [13]:
# Save state_stats as bucketed table
state_stats_bucketed = "state_stats_bucketed"

state_stats.write \
    .mode("overwrite") \
    .bucketBy(20, "state") \
    .sortBy("state") \
    .saveAsTable(state_stats_bucketed)

print(f"State stats bucketed table created: {state_stats_bucketed}")

State stats bucketed table created: state_stats_bucketed


In [14]:
# JOIN TEST: Bucketed vs Non-Bucketed
print("="*80)
print("JOIN PERFORMANCE TEST")
print("="*80)

# Non-bucketed join
print("\n1. Non-bucketed join:")
start_time = time.time()
result_non_bucketed = df.join(state_stats, on="state", how="inner").count()
time_non_bucketed = time.time() - start_time
print(f"   Result: {result_non_bucketed:,} records")
print(f"   Time: {time_non_bucketed:.2f} seconds")

# Bucketed join
print("\n2. Bucketed join:")
df_bucketed_reload = spark.table(bucketed_table)
state_stats_bucketed_reload = spark.table(state_stats_bucketed)

start_time = time.time()
result_bucketed = df_bucketed_reload.join(state_stats_bucketed_reload, on="state", how="inner").count()
time_bucketed = time.time() - start_time
print(f"   Result: {result_bucketed:,} records")
print(f"   Time: {time_bucketed:.2f} seconds")

speedup = time_non_bucketed / time_bucketed if time_bucketed > 0 else 0
print(f"\n⚡ Speedup: {speedup:.2f}x faster with bucketing!")
print("\n✅ Bucketing avoids shuffle during join because data is pre-distributed!")

JOIN PERFORMANCE TEST

1. Non-bucketed join:
   Result: 1,073,741 records
   Time: 0.56 seconds

2. Bucketed join:
   Result: 1,073,741 records
   Time: 0.21 seconds

⚡ Speedup: 2.75x faster with bucketing!

✅ Bucketing avoids shuffle during join because data is pre-distributed!


## 8. Combining Partitioning and Bucketing

In [15]:
# You can combine both!
combined_table = "indian_citizens_partitioned_and_bucketed"

print("Writing data with both partitioning and bucketing...")
start_time = time.time()

df.write \
    .mode("overwrite") \
    .partitionBy("gender") \
    .bucketBy(10, "state") \
    .sortBy("state") \
    .saveAsTable(combined_table)

write_time = time.time() - start_time
print(f"Combined partitioning and bucketing completed in {write_time:.2f} seconds")
print(f"\nConfiguration:")
print(f"  - Partitioned by: gender")
print(f"  - Bucketed by: state (10 buckets)")
print(f"  - Sorted by: state")

Writing data with both partitioning and bucketing...


Combined partitioning and bucketing completed in 1.95 seconds

Configuration:
  - Partitioned by: gender
  - Bucketed by: state (10 buckets)
  - Sorted by: state


## 9. Best Practices & Recommendations

### When to Use Partitioning:
- ✅ Column has **low cardinality** (e.g., state, date, category)
- ✅ Queries frequently **filter** by this column
- ✅ Data is **time-series** (partition by date/month/year)
- ❌ Avoid high cardinality columns (creates too many directories) ( AS we discussed in last seesion )

### When to Use Bucketing:
- ✅ Performing **frequent joins** on a column
- ✅ Need **consistent file sizes**
- ✅ Column has **high cardinality** (e.g., user_id, product_id)
- ✅ Want to **avoid shuffle** during joins

### Combining Both:
- Partition by **low cardinality** column (e.g., date)
- Bucket by **join key** column (e.g., user_id)
- Example: Partition by date, bucket by user_id

## 10. Summary & Comparison

In [19]:
print("="*80)
print("SUMMARY: PARTITIONING vs BUCKETING")
print("="*80)

summary = """
┌─────────────────────┬──────────────────────────┬──────────────────────────┐
│ Feature             │ Partitioning             │ Bucketing                │
├─────────────────────┼──────────────────────────┼──────────────────────────┤
│ Storage             │ Separate directories     │ Fixed number of files    │
│ Best for            │ Filter queries           │ Join operations          │
│ Cardinality         │ Low (10-1000 values)     │ High (1000+ values)      │
│ Partition pruning   │ Yes                      │ No                       │
│ Shuffle avoidance   │ No                       │ Yes (for joins)          │
│ File count          │ Variable (per partition) │ Fixed (num buckets)      │
│ Use case            │ Date-based queries       │ Join-heavy workloads     │
└─────────────────────┴──────────────────────────┴──────────────────────────┘
"""

print(summary)

print("\n✅ Key Takeaways:")
print("1. Partitioning = Faster filters (partition pruning)")
print("2. Bucketing = Faster joins (no shuffle needed)")
print("3. Can combine both for optimal performance")
print("4. Choose based on your query patterns!")

SUMMARY: PARTITIONING vs BUCKETING

┌─────────────────────┬──────────────────────────┬──────────────────────────┐
│ Feature             │ Partitioning             │ Bucketing                │
├─────────────────────┼──────────────────────────┼──────────────────────────┤
│ Storage             │ Separate directories     │ Fixed number of files    │
│ Best for            │ Filter queries           │ Join operations          │
│ Cardinality         │ Low (10-1000 values)     │ High (1000+ values)      │
│ Partition pruning   │ Yes                      │ No                       │
│ Shuffle avoidance   │ No                       │ Yes (for joins)          │
│ File count          │ Variable (per partition) │ Fixed (num buckets)      │
│ Use case            │ Date-based queries       │ Join-heavy workloads     │
└─────────────────────┴──────────────────────────┴──────────────────────────┘


✅ Key Takeaways:
1. Partitioning = Faster filters (partition pruning)
2. Bucketing = Faster joins (no sh

## 11. Cleanup (Optional)

In [20]:
# Uncomment to drop tables and clean up
spark.sql(f"DROP TABLE IF EXISTS {bucketed_table}")
spark.sql(f"DROP TABLE IF EXISTS {state_stats_bucketed}")
spark.sql(f"DROP TABLE IF EXISTS {combined_table}")
print("Tables dropped")

Tables dropped


In [21]:
# Stop Spark session
spark.stop()
print("Spark session stopped")

Spark session stopped
